# 05 — Agent v1 vs v2

The first full eval run showed where the original agent (v1) goes wrong:

- it executed trades without waiting for the user to confirm
- it said "let me look that up" and stopped without calling a tool
- it picked a tool that could not answer the question (today's change for a
  three-month question, average cost for the latest purchase price)
- it asked the user for data a tool could provide, or made a number up

v2 (`agent/prompts.py`) addresses these with a rewritten system prompt, clearer
tool descriptions, and a code-level confirmation gate: a trade only executes if
the model proposed the same trade in an earlier turn.

This notebook runs both versions on both models with the same scorer. v2 was
written after reading v1 failures from across the whole suite, so the v2 score
is not a held-out number. The summary reports the two halves of the suite
separately to show the gain is not concentrated in a few questions.

In [ ]:
# --- Setup: get the code and install the agent-side packages ---
REPO_URL = "https://github.com/sauerpatchkid/Portfolio_Assistant.git"
REPO_DIR = "/content/portfolio-assistant"

import os

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
!pip install -q -r requirements.txt

os.makedirs("results", exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- Serving environment: vLLM + LMCache in their own virtualenv ---
# The server runs as a separate process, so it gets a separate environment.
# That keeps its packages from clashing with the ones Colab ships in the notebook.
VENV = "/content/venv-vllm"

!pip install -q uv
if not os.path.exists(VENV):
    !uv venv {VENV} --python 3.12 -q
    !uv pip install -q --python {VENV}/bin/python vllm lmcache
os.environ["SERVING_VENV"] = VENV

# Record the exact versions so the results can be reproduced.
!uv pip freeze --python {VENV}/bin/python | grep -iE "^(vllm|lmcache|torch|transformers)==" | tee results/versions_vllm.txt

In [ ]:
from serving.launch import start_server, stop_server

LARGE_MODEL = "Qwen/Qwen3-8B-AWQ"
SMALL_MODEL = "Qwen/Qwen3-0.6B"

## Large model

In [ ]:
server = start_server("vllm", LARGE_MODEL, log_name="agent_versions_large")
for version in ["v1", "v2"]:
    !python -m eval.run_eval --name agent_{version}_8b --model {LARGE_MODEL} --prompt {version} --concurrency 16
stop_server(server)

## Small model

In [ ]:
server = start_server("vllm", SMALL_MODEL, log_name="agent_versions_small")
for version in ["v1", "v2"]:
    !python -m eval.run_eval --name agent_{version}_0.6b --model {SMALL_MODEL} --prompt {version} --concurrency 16
stop_server(server)

## Results

In [ ]:
import json
import pandas as pd

summary = pd.read_csv("results/summary.csv")
display(summary[["name", "prompt", "correct", "accuracy_pct", "dev_correct", "test_correct",
                 "multi_step_correct", "errors", "wall_s"]])

# Correct answers per category, one column per run
rows = {}
for name in summary["name"]:
    for line in open(f"results/{name}/records.jsonl"):
        record = json.loads(line)
        rows.setdefault(record["category"], {}).setdefault(name, 0)
        rows[record["category"]][name] += int(record["correct"])
pd.DataFrame(rows).T

In [ ]:
# --- Download the results ---
!zip -qr /content/results_05_agent_versions.zip results
from google.colab import files
files.download("/content/results_05_agent_versions.zip")